# ⚽ FC Barcelona - More Than A Hack 2027
## Interactive Spatio-Temporal Possession & Tactic Analysis

This interactive notebook demonstrates our **Spatio-Temporal Pipeline** for frame-by-frame ball possession classification.
- **Match:** FC Barcelona vs Athletic Club (LaLiga 2024-25, Matchday 2)
- **Data:** 199,319 frames at 25 FPS (Tactical ATD, Smart Tagging, Homography)
- **Metric:** `spellF1` (Temporal IoU matching over possession spells) & `macroF1`

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.config import PITCH_LENGTH, PITCH_WIDTH, TOTAL_FRAMES
from starter.utils import draw_pitch
print('Environment initialized successfully!')

### 1. Load Spatio-Temporal Tracking Features
We extract ball positions, player speeds, and distances using our streaming parser in `src.data_loader`.

In [ ]:
from src.data_loader import stream_tracking_features, load_smart_tagging_intervals

feats = stream_tracking_features()
print(f'Total frames loaded: {len(feats["ball_visible"])}')
print(f'Ball visible frames: {np.sum(feats["ball_visible"])} ({np.mean(feats["ball_visible"])*100:.1f}%)')

### 2. Visualizing Match Stoppage: Collective Velocity Droop
When the referee stops play for a foul or set-piece, all players slow down to a walk (< 1.55 m/s).

In [ ]:
# Plot mean player velocity across the first 5 minutes of play (Frames 11,185 to 18,684)
start_f, end_f = 11185, 18684
frames = np.arange(start_f, end_f)
speeds = feats['mean_player_speed'][start_f:end_f]

plt.figure(figsize=(14, 4))
plt.plot((frames - start_f)/25.0, speeds, color='#1f77b4', lw=1.5, label='Mean Player Speed (m/s)')
plt.axhline(1.55, color='red', linestyle='--', label='Stoppage Threshold (1.55 m/s)')

# Mark the famous foul at frame 15889..16399 (03:07 to 03:27)
foul_start = (15889 - start_f)/25.0
foul_end = (16399 - start_f)/25.0
plt.axvspan(foul_start, foul_end, color='orange', alpha=0.3, label='Verified Foul Stoppage (GT=D)')

plt.title('Collective Player Velocity Droop During Play vs Stoppage', fontsize=13)
plt.xlabel('Time into Match (seconds)', fontsize=11)
plt.ylabel('Speed (m/s)', fontsize=11)
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.show()

### 3. Official Evaluation Scoreboard
Let's inspect the official scoreboard generated by `evaluation/run_evaluation.py`.

In [ ]:
scoreboard_path = os.path.join(ROOT, 'evaluation', 'scoreboard.csv')
if os.path.exists(scoreboard_path):
    df_board = pd.read_csv(scoreboard_path)
    display(df_board[['submission', 'macroF1', 'spellF1', 'poss', 'dur', 'frag']])
else:
    print('Run python evaluation/run_evaluation.py to produce the scoreboard.')